In [ ]:
import urllib.request

urllib.request.urlretrieve('https://www.statmt.org/europarl/v7/fr-en.tgz', 'fr-en.tgz')
urllib.request.urlretrieve('https://www.statmt.org/europarl/v7/es-en.tgz', 'es-en.tgz')

('es-en.tgz', <http.client.HTTPMessage at 0x7e35e61a8770>)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:

import tarfile
import os

dst_folder = "dataset"
file_path = "fr-en.tgz"


MODEL_PATH = "/content/drive/MyDrive/nmt/model.keras" # Changed to .keras extension
TOKENIZER_PATH = "/content/drive/MyDrive/nmt/tokenizer.pkl"
STATE_PATH = "/content/drive/MyDrive/nmt/state.pkl"

if not os.path.exists(dst_folder):
        os.makedirs(dst_folder)



try:
  with tarfile.open(file_path, "r:gz") as tar:
    tar.extractall(path=dst_folder)
  print(f"Successfully extract file {file_path} to {dst_folder}")
except tarfile.TarError as e:
  print(f"An error occurred during extraction: {e}")
except FileNotFoundError:
  print(f"File {file_path} not found.")

dst_folder = "dataset"
file_path = "es-en.tgz"

if not os.path.exists(dst_folder):
        os.makedirs(dst_folder)



try:
  with tarfile.open(file_path, "r:gz") as tar:
    tar.extractall(path=dst_folder)
  print(f"Successfully extract file {file_path} to {dst_folder}")
except tarfile.TarError as e:
  print(f"An error occurred during extraction: {e}")
except FileNotFoundError:
  print(f"File {file_path} not found.")

/tmp/ipython-input-1102508694.py:19: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=dst_folder)


Successfully extract file fr-en.tgz to dataset


/tmp/ipython-input-1102508694.py:36: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=dst_folder)


Successfully extract file es-en.tgz to dataset


In [ ]:
with open(dst_folder + "/europarl-v7.fr-en.en", "r", encoding="utf-8") as en:
  en_fr_content = en.readlines()
with open(dst_folder + "/europarl-v7.fr-en.fr", "r", encoding="utf-8") as fr:
  fr_en_content = fr.readlines()
with open(dst_folder + "/europarl-v7.es-en.es", "r", encoding="utf-8") as es:
  es_en_content = es.readlines()
with open(dst_folder + "/europarl-v7.es-en.en", "r", encoding="utf-8") as en:
  en_es_content = en.readlines()

In [ ]:
import re
from nltk.tokenize import word_tokenize
import nltk

nltk.download('punkt_tab')

def clean(line):
    line = line.strip()

    line = re.sub(r'<.*?>', '', line)

    line = re.sub(r'[^a-zA-Z0-9\s]', '', line)

    line = re.sub(r'\d+', '', line)

    line = line.lower()

    return line

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
en_fr_list = [clean(line) for line in en_fr_content]
fr_en_list = [clean(line) for line in fr_en_content]
es_en_list = [clean(line) for line in es_en_content]
en_es_list = [clean(line) for line in en_es_content]

In [ ]:
pairs = []

# EN → FR
for en, fr in zip(en_fr_list, fr_en_list):
    src = "<en> <to_fr> " + en
    tgt = "<sos> " + fr + " <eos>"
    pairs.append((src, tgt))

# FR → EN
for en, fr in zip(en_fr_list, fr_en_list):
    src = "<fr> <to_en> " + fr
    tgt = "<sos> " + en + " <eos>"
    pairs.append((src, tgt))

# EN -> ES
for en, es in zip(en_es_list, es_en_list):
    src = "<en> <to_es> " + en
    tgt = "<sos> " + es + " <eos>"
    pairs.append((src, tgt))

# ES -> EN
for en, es in zip(en_es_list, es_en_list):
    src = "<es> <to_en> " + es
    tgt = "<sos> " + en + " <eos>"
    pairs.append((src, tgt))

In [ ]:
from sklearn.model_selection import train_test_split

train_pairs, test_pairs = train_test_split(pairs, test_size=0.2, random_state=42)

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
import os
import pickle

all_texts = [p[0] for p in pairs] + [p[1] for p in pairs]

tokenizer = Tokenizer(num_words=30000, filters='')
tokenizer.fit_on_texts(all_texts)

# Ensure the directory exists before saving the tokenizer
os.makedirs(os.path.dirname(TOKENIZER_PATH), exist_ok=True)

with open(TOKENIZER_PATH, "wb") as f:
    pickle.dump(tokenizer, f)

In [ ]:
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.models import Model
import pickle

with open(TOKENIZER_PATH, "rb") as f:
    tokenizer = pickle.load(f)

vocab_size = len(tokenizer.word_index) + 1
latent_dim = 128
encoder_inputs = Input(shape=(None,))
encoder_emb = Embedding(vocab_size, latent_dim, mask_zero=True)(encoder_inputs)
_, state_h, state_c = LSTM(latent_dim, return_state=True)(encoder_emb)

encoder_states = [state_h, state_c]

decoder_inputs = Input(shape=(None,))
decoder_emb = Embedding(vocab_size, latent_dim, mask_zero=True)(decoder_inputs)

decoder_outputs, _, _ = LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True
)(decoder_emb, initial_state=encoder_states)

decoder_outputs = Dense(vocab_size, activation='softmax')(decoder_outputs)

In [ ]:
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_3       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, None, 128) │ 42,721,792 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_2         │ (None, None)      │          0 │ input_layer_2[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, None, 128) │ 42,721,792 │ input_layer_3[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ [(None, 128),     │    131,584 │ embedding_2[0][0… │
│                     │ (None, 128),      │            │ not_equal_2[0][0] │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_3 (LSTM)       │ [(None, None,     │    131,584 │ embedding_3[0][0… │
│                     │ 128), (None,      │            │ lstm_2[0][1],     │
│                     │ 128), (None,      │            │ lstm_2[0][2]      │
│                     │ 128)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, None,      │ 43,055,556 │ lstm_3[0][0]      │
│                     │ 333764)           │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 128,762,308 (491.19 MB)

 Trainable params: 128,762,308 (491.19 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
import os, gc
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences









# model = load_model(MODEL_PATH)


def chunked(data, size):
    for i in range(0, len(data), size):
        yield data[i:i + size]



def save_state(chunk_id):
    with open(STATE_PATH, "wb") as f:
        pickle.dump({"chunk_id": chunk_id}, f)

def load_state():
    if os.path.exists(STATE_PATH):
        with open(STATE_PATH, "rb") as f:
            return pickle.load(f)["chunk_id"]
    return 0



MAX_LEN = 40
CHUNK_SIZE = 10000
BATCH_SIZE = 16
EPOCHS_PER_CHUNK = 1


start_chunk = load_state()
print("Resuming from chunk:", start_chunk)

# --- FIX: Change MODEL_PATH to use .h5 extension for compatibility with Google Drive ---
MODEL_PATH = "/content/drive/MyDrive/nmt/model.h5"

for chunk_id, chunk_pairs in enumerate(chunked(train_pairs, CHUNK_SIZE)):

    if chunk_id < start_chunk:
        continue   # skip already-trained chunks

    print(f"\nTraining chunk {chunk_id}")

    # Prepare data
    encoder_texts = [p[0] for p in chunk_pairs]
    decoder_texts = [p[1] for p in chunk_pairs]

    encoder_seq = tokenizer.texts_to_sequences(encoder_texts)
    decoder_seq = tokenizer.texts_to_sequences(decoder_texts)

    encoder_seq = pad_sequences(encoder_seq, maxlen=MAX_LEN, padding="post")
    decoder_seq = pad_sequences(decoder_seq, maxlen=MAX_LEN, padding="post")

    decoder_in  = decoder_seq[:, :-1]
    decoder_out = decoder_seq[:, 1:]
    decoder_out = np.expand_dims(decoder_out, -1)

    # Train
    model.fit(
        [encoder_seq, decoder_in],
        decoder_out,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS_PER_CHUNK
    )

    # Save model + state
    model.save(MODEL_PATH)
    save_state(chunk_id + 1)

    # Free memory
    del encoder_seq, decoder_seq, decoder_in, decoder_out
    gc.collect()

Resuming from chunk: 0

Training chunk 0
625/625 ━━━━━━━━━━━━━━━━━━━━ 281s 449ms/step - accuracy: 0.1421 - loss: 4.9845



Training chunk 1
625/625 ━━━━━━━━━━━━━━━━━━━━ 287s 459ms/step - accuracy: 0.1434 - loss: 4.9183



Training chunk 2
625/625 ━━━━━━━━━━━━━━━━━━━━ 289s 462ms/step - accuracy: 0.1482 - loss: 4.8514



Training chunk 3
625/625 ━━━━━━━━━━━━━━━━━━━━ 288s 461ms/step - accuracy: 0.1492 - loss: 4.7983



Training chunk 4
625/625 ━━━━━━━━━━━━━━━━━━━━ 289s 462ms/step - accuracy: 0.1524 - loss: 4.7640



Training chunk 5
 81/625 ━━━━━━━━━━━━━━━━━━━━ 4:11 463ms/step - accuracy: 0.1570 - loss: 4.7247

In [ ]:
from tensorflow.keras.models import load_model

# Ensure MODEL_PATH is set to the .h5 file
MODEL_PATH = "/content/drive/MyDrive/nmt/model.h5"

# Load the model
try:
    loaded_model = load_model(MODEL_PATH)
    print("Model loaded successfully!")
    loaded_model.summary()
except Exception as e:
    print(f"Error loading model: {e}")